In [1]:
!pip install fastprop

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.0/846.0 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.0/176.0 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.5/849.5 kB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.9/20.9 MB 124.0 MB/s eta 0:00:00


In [2]:
# @title train Model
!fastprop train \
  --input-file train.csv \
  --smiles-column SMILES \
  --target-columns LogS \
  --output-directory fastprop_results \
  --problem-type regression \
  --number-epochs 100 \
  --batch-size 256

Seed set to 42
[12/15/2025 10:08:09 AM fastprop.descriptors] INFO: Calculating descriptors
100% 7479/7479 [11:36<00:00, 10.73it/s]
[12/15/2025 10:19:48 AM fastprop.descriptors] INFO: Descriptor calculation complete, elapsed time: 0:11:39.757102
[12/15/2025 10:20:05 AM fastprop.descriptors] INFO: Cached descriptors to fastprop_results/cached_train_all_1765793225.csv
[12/15/2025 10:20:05 AM fastprop.cli.train] INFO: Run 'tensorboard --logdir fastprop_results/fastprop_1765793287/tensorboard_logs' to track training progress.
[12/15/2025 10:20:05 AM fastprop.cli.train] INFO: Training model 1 of 1 (random_seed=42)
[12/15/2025 10:20:06 AM fastprop.cli.train] INFO: Model architecture:
{fastprop(
  (fnn): Sequential(
    (lin1): Linear(in_features=1613, out_features=1800, bias=True)
    (act1): ReLU()
    (lin2): Linear(in_features=1800, out_features=1800, bias=True)
  )
  (readout): Linear(in_features=1800, out_features=1, bias=True)
)}
GPU available: True (cuda), used: True
TPU available: Fal

In [3]:
# @title run predictions
import os
import pandas as pd

# A. Create a temporary SMILES file from your benchmark file
# We assume 'Tight_set.csv' is your SC2 benchmark file
test_df = pd.read_csv("sc2.csv")
test_df["SMILES"].to_csv("test_smiles.txt", index=False, header=False)

# B. Find the trained model folder (Fastprop adds a timestamp)
base_dir = "fastprop_results"
# Find the latest directory created inside fastprop_results
latest_run = max([os.path.join(base_dir, d) for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))], key=os.path.getmtime)
checkpoint_dir = os.path.join(latest_run, "checkpoints")

print(f"Using model from: {checkpoint_dir}")

# C. Run Prediction using the bash command
# We use $checkpoint_dir to pass the python variable to the shell
!fastprop predict \
  -sf test_smiles.txt \
  -ds all \
  -o fastprop_predictions.csv \
  $checkpoint_dir

Using model from: fastprop_results/fastprop_1765793287/checkpoints
100% 100/100 [00:15<00:00,  6.45it/s]
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Predicting ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1 0:00:00 • 0:00:00 0.00it/s 


In [ ]:
# @title check score
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score

gt_df = pd.read_csv("sc2.csv")
pred_df = pd.read_csv("fastprop_predictions.csv")

gt_df['fastprop_pred'] = pred_df['task_0'].values

target_col = "Solubility" if "Solubility" in gt_df.columns else "LogS"
y_true = gt_df[target_col]
y_pred = gt_df['fastprop_pred']

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print(f"RMSE: {rmse:.4f}")
print(f"R2: {r2:.4f}")


RMSE: 1.0571
R2: 0.3027
----------------------------------
CGBoost RMSE: 0.8400
